# Phase 2 (c): Evaluation & Comparison

This notebook puts everything together:

1. Re-runs all three models on the same train/test split
2. Adds the **Global Average** baseline
3. Builds the comparison table and bar chart
4. Computes **Precision@5** and **Recall@5** (bonus)
5. Saves all figures and the final `results/metrics.csv`
6. Discussion: which model wins and why


In [ ]:
import sys
from pathlib import Path

# Allow `from src.xxx import yyy` when this notebook lives in /notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from surprise import Dataset, Reader, accuracy
from surprise.model_selection import train_test_split as surprise_split

from src.cf_model import tune_user_knn, fit_user_knn
from src.svd_model import tune_svd, fit_svd
from src.evaluation import rmse_mae_constant, precision_recall_at_k

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42


## 1. Load data and build the shared train/test split

In [ ]:
ratings = pd.read_csv("data/processed/ratings_clean.csv")
model_data = ratings[["userId", "movieId", "movieRating"]]

reader = Reader(rating_scale=(1, 5))
data   = Dataset.load_from_df(model_data, reader)

trainset, testset = surprise_split(data, test_size=0.2, random_state=RANDOM_STATE)
print(f"Train: {trainset.n_ratings:,}  Test: {len(testset):,}")


## 2. Model 1 — Global Average baseline

In [ ]:
global_mean = trainset.global_mean
print(f"Global mean rating: {global_mean:.4f}")
rmse_global, mae_global = rmse_mae_constant(testset, global_mean)
print(f"Global Average  RMSE: {rmse_global:.4f}   MAE: {mae_global:.4f}")


## 3. Model 2 — User-Based KNN

In [ ]:
knn_params = tune_user_knn(data, cv=3)
print("Best KNN params:", knn_params)

knn = fit_user_knn(trainset, knn_params)
knn_predictions = knn.test(testset)
rmse_knn = accuracy.rmse(knn_predictions)
mae_knn  = accuracy.mae(knn_predictions)


## 4. Model 3 — SVD

In [ ]:
svd_params = tune_svd(data, cv=3)
print("Best SVD params:", svd_params)

svd = fit_svd(trainset, svd_params)
svd_predictions = svd.test(testset)
rmse_svd = accuracy.rmse(svd_predictions)
mae_svd  = accuracy.mae(svd_predictions)


## 5. Bonus: Precision@5 and Recall@5

A movie is *relevant* if its true rating is >= 4.

In [ ]:
precision_knn, recall_knn = precision_recall_at_k(knn_predictions, k=5, threshold=4)
precision_svd, recall_svd = precision_recall_at_k(svd_predictions, k=5, threshold=4)

print(f"KNN  Precision@5: {precision_knn:.4f}   Recall@5: {recall_knn:.4f}")
print(f"SVD  Precision@5: {precision_svd:.4f}   Recall@5: {recall_svd:.4f}")


## 6. Comparison table

In [ ]:
results = pd.DataFrame({
    "Model":       ["Global Average", "User-Based KNN", "SVD"],
    "RMSE":        [rmse_global,      rmse_knn,         rmse_svd],
    "MAE":         [mae_global,       mae_knn,          mae_svd],
    "Precision@5": [np.nan,           precision_knn,    precision_svd],
    "Recall@5":    [np.nan,           recall_knn,       recall_svd],
})
results.to_csv("results/metrics.csv", index=False)
results


## 7. Comparison bar chart (RMSE / MAE)

In [ ]:
ax = results.set_index("Model")[["RMSE", "MAE"]].plot(
    kind="bar",
    figsize=(8, 5),
    edgecolor="black",
)
ax.set_title("Model Performance Comparison (lower is better)")
ax.set_ylabel("Error")
ax.set_xlabel("Model")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("results/figures/model_comparison.png", dpi=150)
plt.show()


## 8. Discussion: which model wins, and why?

**Best model: SVD (Matrix Factorization).**

- **Global Average** sets the floor. It ignores all user/movie information and
  predicts the same value for everyone — any real recommender must beat it.

- **User-Based KNN** improves on the baseline by leveraging similarities between
  users, but it struggles on this dataset. The rating matrix is **>99.97%
  sparse**, so most pairs of users share very few rated movies in common. With
  so little overlap, similarity estimates are noisy and unreliable.

- **SVD (Matrix Factorization)** typically wins. Instead of comparing raw rating
  vectors, it learns dense, low-dimensional **latent factor** embeddings for
  every user and movie. Every observed rating contributes to refining the
  entire factor space, which generalizes much better than per-pair similarity
  on sparse data. Regularization (`reg_all`) prevents overfitting on the
  long-tail of users/movies with very few ratings.

**Hyperparameter tuning** confirmed that SVD prefers a moderate number of
latent factors plus moderate regularization. For KNN, tuning `k` made only a
small difference — once data is too sparse, no amount of neighbor tuning fixes
the underlying problem.

**Top-K metrics (bonus).** Precision@5 and Recall@5 measure whether the top-5
recommendations are actually movies the user rated highly. These are more
aligned with real recommendation quality than RMSE/MAE, because in production
a system shows users a short list — accuracy on every (user, movie) pair
matters less than getting the top of the list right. SVD also wins here, for
the same reason it wins on RMSE: better generalization on sparse data.

**Takeaway.** Matrix factorization is the right tool for sparse rating data
like CiaoDVD. KNN can be effective on denser datasets but is fundamentally
limited by user/movie overlap. The Global Average baseline shows just how
much value the more sophisticated models add over a constant prediction.
